In [ ]:
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model
from utils import load_memory, save_memory, load_selected_files
import json

llm = init_chat_model(
    model="openai/gpt-oss-120b",
    model_provider="groq", 
)
# llm = ChatGroq(
#     model="openai/gpt-oss-120b",
#     api_key="YOUR_GROQ_API_KEY"
# )

memory_path = "memories/mapping/chat.json"
memory = load_memory(memory_path)

# Load mapping
with open("mapping.json", "r", encoding="utf-8") as f:
    mapping = json.load(f)


def select_files(question: str):

    prompt = f"""
You are a file selector.

Given this mapping:

{json.dumps(mapping, ensure_ascii=False)}

User question:
{question}

Return ONLY a JSON list of filenames that are relevant.
Example:
["file1.md", "file2.md"]
"""

    response = llm.invoke(prompt)

    try:
        return json.loads(response.content)
    except:
        return []


def chat(user_question: str):

    selected_files = select_files(user_question)

    context = load_selected_files(selected_files)

    messages = []

    messages.append({
        "role": "system",
        "content": f"You are a helpful assistant. Use this context:\n{context}"
    })

    messages.extend(memory)

    messages.append({"role": "user", "content": user_question})

    response = llm.invoke(messages)

    memory.append({"role": "user", "content": user_question})
    memory.append({"role": "assistant", "content": response.content})

    save_memory(memory_path, memory)

    return response.content


# Example
print(chat("Explain the most important concept in the docs"))